# Zero-Trust Network Access (ZTNA) — Policy Engine Demo
**Student:** Tarika Bhiogade  
**Institute:** Sandip University  
**Submission date:** 23 October 2026

This notebook demonstrates least-privilege access decisions, simulated MFA/device checks, and JSON-lines audit logging. It is an educational simulation, not a production gateway.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import List, Dict, Any

AUDIT_LOG = Path('ztna_audit.log')
POLICIES = {
    'admin-console': {'allowed_users': {'admin', 'security-analyst'}, 'require_mfa': True, 'require_compliant_device': True},
    'incident-dashboard': {'allowed_users': {'admin', 'security-analyst', 'auditor'}, 'require_mfa': True, 'require_compliant_device': True},
    'public-status': {'allowed_users': {'admin', 'security-analyst', 'auditor', 'viewer'}, 'require_mfa': False, 'require_compliant_device': False},
}

@dataclass
class AccessRequest:
    username: str
    resource: str
    mfa_verified: bool
    device_compliant: bool
    source_ip: str = '127.0.0.1'

def evaluate_access(request: AccessRequest) -> Dict[str, Any]:
    policy = POLICIES.get(request.resource)
    reasons = []
    if policy is None:
        reasons.append('Unknown resource')
    else:
        if request.username not in policy['allowed_users']:
            reasons.append('User is not allowed for this resource')
        if policy['require_mfa'] and not request.mfa_verified:
            reasons.append('Multi-factor authentication is required')
        if policy['require_compliant_device'] and not request.device_compliant:
            reasons.append('Device does not meet compliance requirements')
    result = {
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'username': request.username, 'resource': request.resource,
        'source_ip': request.source_ip, 'mfa_verified': request.mfa_verified,
        'device_compliant': request.device_compliant,
        'decision': 'ALLOW' if not reasons else 'DENY',
        'reasons': reasons or ['All configured access checks passed']
    }
    with AUDIT_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(result) + '\\n')
    return result

In [ ]:
requests = [
    AccessRequest('admin', 'admin-console', True, True, '10.0.0.10'),
    AccessRequest('viewer', 'admin-console', True, True, '10.0.0.11'),
    AccessRequest('security-analyst', 'incident-dashboard', False, True, '10.0.0.12'),
    AccessRequest('auditor', 'public-status', False, False, '10.0.0.13'),
    AccessRequest('security-analyst', 'admin-console', True, False, '10.0.0.14'),
    AccessRequest('unknown-user', 'private-database', True, True, '10.0.0.15'),
]
for req in requests:
    result = evaluate_access(req)
    print(f"{result['decision']:5} | {req.username:18} | {req.resource:20} | {'; '.join(result['reasons'])}")

## Interpretation
- Requests are evaluated every time; access is not granted just because a user is on a network.
- Administrative resources require an approved role, MFA, and a compliant device.
- Unknown resources are denied by default.
- The audit log records the decision and reasons.

**Important:** MFA and device status are simulated booleans. A real deployment should use an identity provider, actual MFA, device posture checks, HTTPS, protected logs, and a Cloudflare Tunnel/Access configuration reviewed by an administrator.